# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [2]:
# Ваш код
import json
with open('final_hw_data/bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)
print(f"количество messages: {len(messages)}")

with open('final_hw_data/organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)
print(f"количество organizations: {len(organizations)}")

with open('final_hw_data/priority_cases.txt', 'r', encoding='utf-8') as f:
    priority_case_numbers = [line.strip() for line in f if line.strip()] 
print(f"priority_case_numbers: {len(priority_case_numbers)}")

количество messages: 54
количество organizations: 30
priority_case_numbers: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [4]:
# Ваш код

import json


with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

org_by_inn = {org['inn']: org for org in organizations}

print(f"Количество организаций в словаре: {len(org_by_inn)}")

target_inn = "7701234567"
info = org_by_inn.get(target_inn)

if info:
    print(f"\nИнформация по ИНН {target_inn}:")
    for key, value in info.items():
        print(f"  {key}: {value}")
else:
    print(f"\nОрганизация с ИНН {target_inn} не найдена.")

Количество организаций в словаре: 30

Информация по ИНН 7701234567:
  inn: 7701234567
  ogrn: 1027700123456
  name: ООО "Альфа-Строй"
  address: г. Москва, ул. Строителей, д. 15
  region: Москва


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [34]:
# Ваш код

import json

# 1. Загружаем данные из файлов
with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# 2. Создаём словарь org_by_inn с помощью dict comprehension
org_by_inn = {org['inn']: org for org in organizations}

# 3. Для каждого сообщения добавляем поля org_name и region
for msg in messages:
    inn = msg.get('publisher_inn')
    org_data = org_by_inn.get(inn)  # если ИНН не найден, вернёт None
    
    if org_data:
        msg['org_name'] = org_data.get('name')
        msg['region'] = org_data.get('region')
    else:
        msg['org_name'] = None
        msg['region'] = None

# 4. Считаем, сколько сообщений удалось связать с организацией
total_messages = len(messages)
found_messages = sum(1 for msg in messages if msg['org_name'] is not None)
not_found_messages = total_messages - found_messages

# 5. Выводим статистику
print(f"Всего сообщений: {total_messages}")
print(f"Удалось связать с организацией: {found_messages}")
print(f" Не удалось связать: {not_found_messages}")
print(f"Процент успешной связки: {found_messages / total_messages * 100:.1f}%")

# 6. Покажем примеры сообщений (первые 3 успешных и 3 неудачных)
print("\n" + "="*60)
print("ПРИМЕРЫ СООБЩЕНИЙ")
print("="*60)

# Успешные сообщения
print("\n Успешно связанные (первые 3):")
successful = [msg for msg in messages if msg['org_name'] is not None]
for i, msg in enumerate(successful[:3]):
    print(f"\n{i+1}. ИНН: {msg.get('publisher_inn')}")
    print(f"   Организация: {msg['org_name']}")
    print(f"   Регион: {msg['region']}")
    print(f"   Тип сообщения: {msg.get('type')}")
    print(f"   Номер дела: {msg.get('case_number')}")

# Неудачные сообщения
print("\n Неудачно связанные (первые 3):")
unsuccessful = [msg for msg in messages if msg['org_name'] is None]
for i, msg in enumerate(unsuccessful[:3]):
    print(f"\n{i+1}. ИНН: {msg.get('publisher_inn')}")
    print(f"   Организация: {msg['org_name']}")
    print(f"   Тип сообщения: {msg.get('type')}")
    print(f"   Номер дела: {msg.get('case_number')}")

# Посмотреть все ИНН, которые не удалось связать
not_found_inns = set()
for msg in messages:
    if msg['org_name'] is None and msg.get('publisher_inn'):
        not_found_inns.add(msg['publisher_inn'])

print("ИНН, которые не найдены в словаре организаций:")
for inn in sorted(not_found_inns):
    print(f"  - '{inn}'")

Всего сообщений: 54
Удалось связать с организацией: 51
 Не удалось связать: 3
Процент успешной связки: 94.4%

ПРИМЕРЫ СООБЩЕНИЙ

 Успешно связанные (первые 3):

1. ИНН: 7701234567
   Организация: ООО "Альфа-Строй"
   Регион: Москва
   Тип сообщения: Введение процедуры
   Номер дела: А60-12345/2025

2. ИНН: 7702345678
   Организация: ЗАО "Бета-Инвест"
   Регион: Москва
   Тип сообщения: Собрание кредиторов
   Номер дела: А60-56789/2025

3. ИНН: 6658123450
   Организация: ООО "Гамма-Трейд"
   Регион: Свердловская область
   Тип сообщения: Завершение процедуры
   Номер дела: А60-11111/2025

 Неудачно связанные (первые 3):

1. ИНН: 
   Организация: None
   Тип сообщения: Подача заявления
   Номер дела: А60-24680/2025

2. ИНН: 9999999999
   Организация: None
   Тип сообщения: Признание банкротом
   Номер дела: А60-14702/2025

3. ИНН: 7705
   Организация: None
   Тип сообщения: Подача заявления
   Номер дела: А60-12345/2025
ИНН, которые не найдены в словаре организаций:
  - '7705'
  - '99999

### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [35]:
# Ваш код

import json

# 1. Загружаем данные из файлов
with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# 2. Читаем файл priority_cases.txt и создаём множество priority_set
with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    # Читаем все строки, убираем символы перевода строки и пустые строки
    priority_case_numbers = [line.strip() for line in f if line.strip()]
    
priority_set = set(priority_case_numbers)

print("="*60)
print("Множество priority_set:")
print("="*60)
print(f"Количество элементов: {len(priority_set)}")
print(f"Содержимое: {priority_set}")

# 3. Создаём множество message_case_set из номеров дел в messages
# Используем list comprehension + filter для непустых номеров
message_case_set = {msg['case_number'] for msg in messages if msg.get('case_number') and msg['case_number'].strip()}

print("\n" + "="*60)
print("Множество message_case_set:")
print("="*60)
print(f"Количество элементов: {len(message_case_set)}")
print(f"Первые 10 элементов (для примера): {list(message_case_set)[:10]}")

# 4. Находим пересечение — номера дел, которые есть и там, и там
intersection = priority_set & message_case_set

print("\n" + "="*60)
print("ПЕРЕСЕЧЕНИЕ (номера дел в приоритетных И в сообщениях):")
print("="*60)
print(f"Количество пересекающихся дел: {len(intersection)}")
print(f"Список пересекающихся дел: {sorted(intersection)}")

# 5. Дополнительно: посмотрим, какие приоритетные дела НЕ нашли в сообщениях
missing_in_messages = priority_set - message_case_set
print("\n" + "="*60)
print("Приоритетные дела, НЕ найденные в сообщениях:")
print("="*60)
print(f"Количество: {len(missing_in_messages)}")
print(f"Список: {sorted(missing_in_messages)}")

# 6. И какие номера из сообщений есть в приоритетных (альтернативный вывод)
print("\n" + "="*60)
print("Детальный вывод пересечения:")
print("="*60)
for case_number in sorted(intersection):
    # Найдём пример сообщения с этим номером дела
    sample_msg = next((msg for msg in messages if msg.get('case_number') == case_number), None)
    if sample_msg:
        print(f"\n Номер дела: {case_number}")
        print(f"   Тип сообщения: {sample_msg.get('type')}")
        print(f"   ИНН публикатора: {sample_msg.get('publisher_inn')}")
        if sample_msg.get('org_name'):
            print(f"   Организация: {sample_msg.get('org_name')}")

Множество priority_set:
Количество элементов: 10
Содержимое: {'А60-44444/2025', 'А60-66666/2025', 'А60-88888/2025', 'А60-99999/2025', 'А60-33333/2025', 'А60-77777/2025', 'А60-12345/2025', 'А60-22222/2025', 'А60-56789/2025', 'А60-11111/2025'}

Множество message_case_set:
Количество элементов: 16
Первые 10 элементов (для примера): ['А60-55555/2025', 'А60-44444/2025', 'А60-25814/2025', 'А60-36925/2025', 'А60-14702/2025', 'А60-66666/2025', 'А60-13579/2025', 'А60-88888/2025', 'А60-99999/2025', 'А60-33333/2025']

ПЕРЕСЕЧЕНИЕ (номера дел в приоритетных И в сообщениях):
Количество пересекающихся дел: 10
Список пересекающихся дел: ['А60-11111/2025', 'А60-12345/2025', 'А60-22222/2025', 'А60-33333/2025', 'А60-44444/2025', 'А60-56789/2025', 'А60-66666/2025', 'А60-77777/2025', 'А60-88888/2025', 'А60-99999/2025']

Приоритетные дела, НЕ найденные в сообщениях:
Количество: 0
Список: []

Детальный вывод пересечения:

 Номер дела: А60-11111/2025
   Тип сообщения: Завершение процедуры
   ИНН публикатора:

---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [13]:
# Ваш код

from datetime import datetime
import re

def parse_date(date_str):
    """
    Парсит строку с датой в разных форматах и возвращает объект datetime.
    
    Поддерживаемые форматы:
    - "DD.MM.YYYY" (например, "15.01.2025")
    - "YYYY-MM-DD" (например, "2025-02-10")
    - "DD месяца YYYY г." (например, "15 марта 2025 г.")
    - "DD/MM/YYYY HH:MM" (например, "15/03/2025 10:30")
    
    Args:
        date_str: строка с датой или None
    
    Returns:
        datetime объект или None, если распарсить не удалось
    """
    
    # Обрабатываем None и пустые строки
    if not date_str or not isinstance(date_str, str):
        return None
    
    # Убираем лишние пробелы
    date_str = date_str.strip()
    
    if not date_str:
        return None
    
    # Словарь для замены русских названий месяцев на цифры
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    # Пробуем разные форматы
    try:
        # Формат 1: "DD.MM.YYYY"
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        # Формат 2: "YYYY-MM-DD"
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        # Формат 3: "DD месяца YYYY г." (с русскими месяцами)
        # Пример: "15 марта 2025 г."
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        # Формат 4: "DD/MM/YYYY HH:MM"
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        # Если ни один формат не подошёл
        return None
        
    except (ValueError, TypeError):
        return None


# ============================================
# Тестирование функции на примерах из файла
# ============================================

# Тестовые строки из bankruptcy_messages.json
test_dates = [
    "15.01.2025",           # формат DD.MM.YYYY
    "2025-02-10",           # формат YYYY-MM-DD
    "2025-03-15T00:00:00",  # ISO с T (не наш формат, но проверим)
    "20 марта 2025 г.",     # формат с русским месяцем
    "2025-04-05",           # формат YYYY-MM-DD
    "10.02.2025",           # формат DD.MM.YYYY
    "15/03/2025 10:30",     # формат DD/MM/YYYY HH:MM
    "ДАТА УТЕРЯНА",         # некорректная дата
    "",                     # пустая строка
    None,                   # None
]

print("="*60)
print("ТЕСТИРОВАНИЕ ФУНКЦИИ parse_date")
print("="*60)

for date_str in test_dates:
    result = parse_date(date_str)
    print(f"Вход: '{date_str}'")
    print(f"Результат: {result}")
    if result:
        print(f"  (тип: {type(result).__name__})")
    print("-"*40)

ТЕСТИРОВАНИЕ ФУНКЦИИ parse_date
Вход: '15.01.2025'
Результат: 2025-01-15 00:00:00
  (тип: datetime)
----------------------------------------
Вход: '2025-02-10'
Результат: 2025-02-10 00:00:00
  (тип: datetime)
----------------------------------------
Вход: '2025-03-15T00:00:00'
Результат: None
----------------------------------------
Вход: '20 марта 2025 г.'
Результат: 2025-03-20 00:00:00
  (тип: datetime)
----------------------------------------
Вход: '2025-04-05'
Результат: 2025-04-05 00:00:00
  (тип: datetime)
----------------------------------------
Вход: '10.02.2025'
Результат: 2025-02-10 00:00:00
  (тип: datetime)
----------------------------------------
Вход: '15/03/2025 10:30'
Результат: 2025-03-15 10:30:00
  (тип: datetime)
----------------------------------------
Вход: 'ДАТА УТЕРЯНА'
Результат: None
----------------------------------------
Вход: ''
Результат: None
----------------------------------------
Вход: 'None'
Результат: None
----------------------------------------


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [36]:
# Ваш код

import re
from datetime import datetime

def parse_date(date_str):
    """
    Парсит строку с датой в разных форматах.
    (та же функция, что мы написали ранее)
    """
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        # Формат 1: "DD.MM.YYYY"
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        # Формат 2: "YYYY-MM-DD"
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        # Формат 2b: "YYYY-MM-DDTHH:MM:SS"
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        # Формат 3: "DD месяца YYYY г."
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        # Формат 4: "DD/MM/YYYY HH:MM"
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def validate_message(msg):
    """
    Проверяет сообщение на валидность.
    
    Args:
        msg: словарь с данными сообщения
    
    Returns:
        tuple: (valid_msg, errors)
        - valid_msg: очищенный словарь или None, если есть критические ошибки
        - errors: список строк с описанием ошибок
    """
    
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    # 1. Проверяем наличие обязательных полей
    for field in required_fields:
        if field not in msg:
            errors.append(f"Отсутствует обязательное поле: '{field}'")
        elif msg[field] is None:
            errors.append(f"Поле '{field}' имеет значение None")
    
    # Если есть критические ошибки (отсутствуют поля), возвращаем None
    if errors:
        return (None, errors)
    
    # Создаём копию для очищенных данных
    valid_msg = msg.copy()
    
    # 2. Проверяем ИНН (должен быть строкой из 10 или 12 цифр)
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    
    if not inn:
        errors.append("Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ИНН '{inn}' имеет неверную длину (должен быть 10 или 12 цифр, получено {len(inn)})")
    else:
        # ИНН валидный, оставляем как есть (уже строка)
        pass
    
    # 3. Проверяем msg_text (непустая строка)
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # 4. Проверяем date_published через parse_date
    date_str = msg.get('date_published')
    parsed_date = parse_date(date_str)
    
    if parsed_date is None:
        errors.append(f"Не удалось распарсить дату: '{date_str}'")
    else:
        valid_msg['parsed_date'] = parsed_date
        # Опционально: заменим исходную дату на парснутую (но лучше сохранить обе)
        # valid_msg['date_published'] = parsed_date
    
    # 5. Проверяем type (непустая строка)
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # 6. Проверяем case_number (непустая строка)
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    # Возвращаем результат
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# ТЕСТИРОВАНИЕ ФУНКЦИИ
# ============================================

# Загружаем данные
import json

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

print("="*60)
print("ТЕСТИРОВАНИЕ validate_message")
print("="*60)

# Тестируем на всех сообщениях
valid_count = 0
invalid_count = 0
all_errors = []

for i, msg in enumerate(messages[:10]):  # Проверим первые 10 сообщений
    print(f"\n Сообщение {i+1}:")
    print(f"   ИНН: {msg.get('publisher_inn')}")
    print(f"   Тип: {msg.get('type')}")
    print(f"   Номер дела: {msg.get('case_number')}")
    
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        print(f" Валидное сообщение")
        valid_count += 1
        if 'parsed_date' in valid_msg:
            print(f"    Распарсеная дата: {valid_msg['parsed_date']}")
    else:
        print(f"  Невалидное сообщение")
        invalid_count += 1
        for error in errors:
            print(f"      - {error}")
            all_errors.append(error)

# Общая статистика
print("\n" + "="*60)
print("СТАТИСТИКА")
print("="*60)
print(f" Валидных сообщений: {valid_count}")
print(f" Невалидных сообщений: {invalid_count}")

# Уникальные типы ошибок
if all_errors:
    print("\n Типы ошибок:")
    from collections import Counter
    error_counts = Counter(all_errors)
    for error, count in error_counts.most_common():
        print(f"   - {error} (встретилось {count} раз)")

ТЕСТИРОВАНИЕ validate_message

 Сообщение 1:
   ИНН: 7701234567
   Тип: Введение процедуры
   Номер дела: А60-12345/2025
 Валидное сообщение
    Распарсеная дата: 2025-01-15 00:00:00

 Сообщение 2:
   ИНН: 7702345678
   Тип: Собрание кредиторов
   Номер дела: А60-56789/2025
 Валидное сообщение
    Распарсеная дата: 2025-02-10 00:00:00

 Сообщение 3:
   ИНН: 6658123450
   Тип: Завершение процедуры
   Номер дела: А60-11111/2025
 Валидное сообщение
    Распарсеная дата: 2025-03-15 00:00:00

 Сообщение 4:
   ИНН: 7810345612
   Тип: Признание банкротом
   Номер дела: А60-22222/2025
 Валидное сообщение
    Распарсеная дата: 2025-03-20 00:00:00

 Сообщение 5:
   ИНН: 5027123456
   Тип: Продажа имущества
   Номер дела: А60-33333/2025
 Валидное сообщение
    Распарсеная дата: 2025-04-05 00:00:00

 Сообщение 6:
   ИНН: 7705456789
   Тип: Введение процедуры
   Номер дела: А60-44444/2025
 Валидное сообщение
    Распарсеная дата: 2025-01-25 00:00:00

 Сообщение 7:
   ИНН: 7448123456
   Тип: Требова

### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [37]:
# Ваш код

import json
from collections import Counter
import re
from datetime import datetime

# ============================================
# 1. Функция parse_date (из предыдущего задания)
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


# ============================================
# 2. Функция validate_message
# ============================================

def validate_message(msg):
    """
    Проверяет сообщение на валидность.
    Возвращает кортеж (valid_msg, errors)
    """
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    # 1. Проверяем наличие обязательных полей
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует поле '{field}'")
        elif msg[field] is None:
            errors.append(f"ValueError: Поле '{field}' имеет значение None")
    
    # Если есть критические ошибки, возвращаем None
    if errors:
        return (None, errors)
    
    # Создаём копию для очищенных данных
    valid_msg = msg.copy()
    
    # 2. Проверяем ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину (должен быть 10 или 12 цифр, получено {len(inn)})")
    else:
        valid_msg['publisher_inn'] = inn
    
    # 3. Проверяем msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # 4. Проверяем date_published
    date_str = msg.get('date_published')
    parsed_date = parse_date(date_str)
    
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату '{date_str}'")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # 5. Проверяем type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # 6. Проверяем case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    # Возвращаем результат
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 3. Загружаем данные и применяем валидацию
# ============================================

# Загружаем сообщения
with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

print("="*70)
print("ВАЛИДАЦИЯ ВСЕХ СООБЩЕНИЙ")
print("="*70)
print(f"Всего сообщений для проверки: {len(messages)}")
print()

# Разделяем результаты
valid_messages = []
error_records = []

for i, msg in enumerate(messages):
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'index': i,
            'original_msg': msg,
            'errors': errors
        })

# ============================================
# 4. Выводим основные результаты
# ============================================

print("="*70)
print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ")
print("="*70)
print(f" Валидных сообщений: {len(valid_messages)}")
print(f" Невалидных сообщений: {len(error_records)}")
print(f" Процент валидных: {len(valid_messages) / len(messages) * 100:.1f}%")

# ============================================
# 5. Собираем статистику ошибок по типам
# ============================================

error_type_counter = Counter()

for error_record in error_records:
    for error_msg in error_record['errors']:
        # Извлекаем тип ошибки (KeyError, ValueError)
        if error_msg.startswith('KeyError'):
            error_type_counter['KeyError'] += 1
        elif error_msg.startswith('ValueError'):
            error_type_counter['ValueError'] += 1
        else:
            error_type_counter['Other'] += 1

print("\n" + "="*70)
print("СТАТИСТИКА ОШИБОК ПО ТИПАМ")
print("="*70)
for error_type, count in error_type_counter.most_common():
    print(f"  {error_type}: {count}")

# ============================================
# 6. Детальная статистика по конкретным ошибкам
# ============================================

detailed_error_counter = Counter()

for error_record in error_records:
    for error_msg in error_record['errors']:
        detailed_error_counter[error_msg] += 1

print("\n" + "="*70)
print("ДЕТАЛЬНАЯ СТАТИСТИКА ОШИБОК")
print("="*70)
for error, count in detailed_error_counter.most_common(10):
    print(f"  {error} (встретилось {count} раз)")

# ============================================
# 7. Показываем примеры валидных сообщений
# ============================================

print("\n" + "="*70)
print("ПРИМЕРЫ ВАЛИДНЫХ СООБЩЕНИЙ (первые 3)")
print("="*70)
for i, msg in enumerate(valid_messages[:3]):
    print(f"\n Валидное сообщение {i+1}:")
    print(f"   ИНН: {msg.get('publisher_inn')}")
    print(f"   Тип: {msg.get('type')}")
    print(f"   Номер дела: {msg.get('case_number')}")
    print(f"   Дата: {msg.get('date_published')}")
    if 'parsed_date' in msg:
        print(f"   Распарсеная дата: {msg['parsed_date']}")
    print(f"   Текст: {msg.get('msg_text')[:80]}...")

# ============================================
# 8. Показываем примеры ошибок
# ============================================

print("\n" + "="*70)
print("ПРИМЕРЫ НЕВАЛИДНЫХ СООБЩЕНИЙ (первые 3)")
print("="*70)
for i, error_record in enumerate(error_records[:3]):
    print(f"\n Невалидное сообщение {i+1}:")
    print(f"   Индекс в исходном массиве: {error_record['index']}")
    print(f"   Ошибки:")
    for error in error_record['errors']:
        print(f"      - {error}")
    print(f"   Исходные данные:")
    print(f"      ИНН: {error_record['original_msg'].get('publisher_inn')}")
    print(f"      Тип: {error_record['original_msg'].get('type')}")
    print(f"      Дата: {error_record['original_msg'].get('date_published')}")

# ============================================
# 9. Сохраняем результаты в файлы 
# ============================================

# Сохраняем валидные сообщения
with open('valid_messages.json', 'w', encoding='utf-8') as f:
    # Преобразуем datetime в строку для JSON
    messages_to_save = []
    for msg in valid_messages:
        msg_copy = msg.copy()
        if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
            msg_copy['parsed_date'] = msg_copy['parsed_date'].isoformat()
        messages_to_save.append(msg_copy)
    json.dump(messages_to_save, f, ensure_ascii=False, indent=2)

# Сохраняем информацию об ошибках
with open('error_records.json', 'w', encoding='utf-8') as f:
    errors_to_save = []
    for err in error_records:
        errors_to_save.append({
            'index': err['index'],
            'errors': err['errors']
        })
    json.dump(errors_to_save, f, ensure_ascii=False, indent=2)

print("\n" + "="*70)
print("ФАЙЛЫ СОХРАНЕНЫ")
print("="*70)
print(" valid_messages.json - сохранены валидные сообщения")
print(" error_records.json - сохранена информация об ошибках")

ВАЛИДАЦИЯ ВСЕХ СООБЩЕНИЙ
Всего сообщений для проверки: 54

РЕЗУЛЬТАТЫ ВАЛИДАЦИИ
 Валидных сообщений: 48
 Невалидных сообщений: 6
 Процент валидных: 88.9%

СТАТИСТИКА ОШИБОК ПО ТИПАМ
  ValueError: 6

ДЕТАЛЬНАЯ СТАТИСТИКА ОШИБОК
  ValueError: Поле 'publisher_inn' пустое (встретилось 1 раз)
  ValueError: Поле 'date_published' имеет значение None (встретилось 1 раз)
  ValueError: Поле 'type' имеет значение None (встретилось 1 раз)
  ValueError: ИНН '7705' имеет неверную длину (должен быть 10 или 12 цифр, получено 4) (встретилось 1 раз)
  ValueError: Не удалось распарсить дату 'ДАТА УТЕРЯНА' (встретилось 1 раз)
  ValueError: Поле 'case_number' пустое или не является строкой (встретилось 1 раз)

ПРИМЕРЫ ВАЛИДНЫХ СООБЩЕНИЙ (первые 3)

 Валидное сообщение 1:
   ИНН: 7701234567
   Тип: Введение процедуры
   Номер дела: А60-12345/2025
   Дата: 15.01.2025
   Распарсеная дата: 2025-01-15 00:00:00
   Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитра...

 Валидн

---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [39]:
# Ваш код

import json

# Загружаем сообщения
with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# Считаем статистику по суммам
messages_with_amounts = 0
total_amounts_found = 0
all_amounts = []

for msg in messages[:20]:  # Проверим первые 20 сообщений
    text = msg.get('msg_text', '')
    amounts = extract_amounts(text)
    
    if amounts:
        messages_with_amounts += 1
        total_amounts_found += len(amounts)
        all_amounts.extend(amounts)
        
        print(f"\n Сообщение от {msg.get('publisher_inn')}:")
        print(f"   Текст: {text[:100]}...")
        print(f"  Найденные суммы: {amounts}")

print("\n" + "="*70)
print("СТАТИСТИКА")
print("="*70)
print(f" Проверено сообщений: 20")
print(f" Сообщений с суммами: {messages_with_amounts}")
print(f" Всего найдено сумм: {total_amounts_found}")
print(f" Уникальные суммы: {set(all_amounts)}")


 Сообщение от 7701234567:
   Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Ива...
  Найденные суммы: ['15 000 000 руб.']

 Сообщение от 7702345678:
   Текст: Уведомление о проведении собрания кредиторов ЗАО "Бета-Инвест". Сумма требований: 8 500 тыс. руб. АС...
  Найденные суммы: ['8 500 тыс. руб.']

 Сообщение от 6658123450:
   Текст: Сообщение о завершении конкурсного производства. Арбитражный управляющий Петров Пётр Петрович. Общая...
  Найденные суммы: ['42 млн руб.']

 Сообщение от 7810345612:
   Текст: Сообщение о признании должника банкротом. ОАО "Дельта-Логистик" признано банкротом. АС г. Санкт-Пете...
  Найденные суммы: ['3 200 000 руб.']

 Сообщение от 5027123456:
   Текст: Уведомление о продаже имущества должника. ООО "Омега-Финанс". Начальная цена: 12 500 тыс. руб. АС Кр...
  Найденные суммы: ['12 500 тыс. руб.']

 Сообщение от 7705456789:
   Текст: Сообщение о введении внешнего управления. ПАО "Сигма-Энерго". Арбит

### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [40]:
# Ваш код

import json

# Загружаем сообщения
with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

# Считаем статистику
total_messages = len(messages)
messages_with_court = 0
messages_without_court = 0

court_mentions_list = []

for msg in messages:
    text = msg.get('msg_text', '')
    has_court = find_court_mentions(text)
    
    if has_court:
        messages_with_court += 1
        court_mentions_list.append(msg)
    else:
        messages_without_court += 1

print(f"\n Статистика по всем {total_messages} сообщениям:")
print(f"    Упоминают арбитражный суд (АС): {messages_with_court}")
print(f"    Не упоминают: {messages_without_court}")
print(f"    Процент упоминаний: {messages_with_court / total_messages * 100:.1f}%")

# Показываем примеры сообщений с упоминанием суда
print("\n" + "="*70)
print("ПРИМЕРЫ СООБЩЕНИЙ С УПОМИНАНИЕМ СУДА (первые 5)")
print("="*70)

for i, msg in enumerate(court_mentions_list[:5]):
    text = msg.get('msg_text', '')
    # Находим часть с упоминанием суда
    if "АС " in text:
        pos = text.find("АС ")
        context = text[pos:pos+50]
    else:
        context = text[:80]
    
    print(f"\n{i+1}. ИНН: {msg.get('publisher_inn')}")
    print(f"   Тип: {msg.get('type')}")
    print(f"   Упоминание: ...{context}...")

# Показываем примеры сообщений БЕЗ упоминания суда
print("\n" + "="*70)
print("ПРИМЕРЫ СООБЩЕНИЙ БЕЗ УПОМИНАНИЯ СУДА (первые 3)")
print("="*70)

messages_without_court_list = [msg for msg in messages if not find_court_mentions(msg.get('msg_text', ''))]
for i, msg in enumerate(messages_without_court_list[:3]):
    text = msg.get('msg_text', '')
    print(f"\n{i+1}. ИНН: {msg.get('publisher_inn')}")
    print(f"   Тип: {msg.get('type')}")
    print(f"   Текст: {text[:80]}..." if len(text) > 80 else f"   Текст: {text}")


 Статистика по всем 54 сообщениям:
    Упоминают арбитражный суд (АС): 46
    Не упоминают: 8
    Процент упоминаний: 85.2%

ПРИМЕРЫ СООБЩЕНИЙ С УПОМИНАНИЕМ СУДА (первые 5)

1. ИНН: 7701234567
   Тип: Введение процедуры
   Упоминание: ...АС Свердловской области....

2. ИНН: 7702345678
   Тип: Собрание кредиторов
   Упоминание: ...АС г. Москвы....

3. ИНН: 7810345612
   Тип: Признание банкротом
   Упоминание: ...АС г. Санкт-Петербурга. Долг 3 200 000 руб....

4. ИНН: 5027123456
   Тип: Продажа имущества
   Упоминание: ...АС Краснодарского края....

5. ИНН: 7705456789
   Тип: Введение процедуры
   Упоминание: ...АС г. Москвы....

ПРИМЕРЫ СООБЩЕНИЙ БЕЗ УПОМИНАНИЯ СУДА (первые 3)

1. ИНН: 6658123450
   Тип: Завершение процедуры
   Текст: Сообщение о завершении конкурсного производства. Арбитражный управляющий Петров ...

2. ИНН: 6160123456
   Тип: Подача заявления
   Текст: Сообщение о подаче заявления о банкротстве. Долг составляет 2 300 000 руб.

3. ИНН: 7705123450
   Тип: Введение проц

### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [41]:
# Ваш код

import json

# ============================================
# 1. Определяем функцию extract_manager_name
# ============================================

def extract_manager_name(text):
    """
    Ищет ФИО арбитражного управляющего в тексте.
    """
    if not text or not isinstance(text, str):
        return None
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return None
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else None


# ============================================
# 2. Загружаем данные
# ============================================

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)


# ============================================
# 3. Применяем функцию
# ============================================

total_messages = len(messages)
messages_with_manager = 0
messages_without_manager = 0
managers_list = []

for msg in messages:
    text = msg.get('msg_text', '')
    manager_name = extract_manager_name(text)
    
    if manager_name:
        messages_with_manager += 1
        managers_list.append(manager_name)
    else:
        messages_without_manager += 1


# ============================================
# 4. Выводим результаты
# ============================================

print("="*60)
print("РЕЗУЛЬТАТЫ")
print("="*60)
print(f"Всего сообщений: {total_messages}")
print(f" Найдено ФИО управляющего: {messages_with_manager}")
print(f" Не найдено: {messages_without_manager}")
if total_messages > 0:
    print(f" Процент: {messages_with_manager / total_messages * 100:.1f}%")

# Показываем найденных управляющих
print("\n" + "="*60)
print("ПРИМЕРЫ НАЙДЕННЫХ УПРАВЛЯЮЩИХ")
print("="*60)
for i, manager in enumerate(managers_list[:10]):
    print(f"{i+1}. {manager}")

РЕЗУЛЬТАТЫ
Всего сообщений: 54
 Найдено ФИО управляющего: 10
 Не найдено: 44
 Процент: 18.5%

ПРИМЕРЫ НАЙДЕННЫХ УПРАВЛЯЮЩИХ
1. Иванов Иван Иванович.
2. Петров Пётр Петрович.
3. Сидоров Сергей Сергеевич.
4. Николаев Николай Николаевич.
5. Фёдоров Фёдор Фёдорович.
6. Алексеев Алексей Алексеевич.
7. Дмитриев Дмитрий Дмитриевич.
8. Сергеев Сергей Сергеевич.
9. Андреев Андрей Андреевич.
10. Васильев Василий Васильевич.


### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [42]:
# Ваш код

import json
import re
from datetime import datetime

# ============================================
# 1. ВСЕ НЕОБХОДИМЫЕ ФУНКЦИИ
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def extract_amounts(text):
    """Ищет упоминания денежных сумм в тексте"""
    if not text or not isinstance(text, str):
        return []
    
    amounts = []
    
    # Паттерны для поиска сумм
    patterns = [
        (r'(\d{1,3}(?:\s\d{3})*)\s+млн\s+руб\.', 'млн руб.'),
        (r'(\d{1,3}(?:\s\d{3})*)\s+тыс\.\s+руб\.', 'тыс. руб.'),
        (r'(\d{1,3}(?:\s\d{3})*(?:\s\d+)?)\s+руб\.', 'руб.')
    ]
    
    for pattern, suffix in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            amounts.append(f"{match} {suffix}")
    
    return amounts


def find_court_mentions(text):
    """Проверяет упоминание арбитражного суда"""
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


def extract_manager_name(text):
    """Ищет ФИО арбитражного управляющего"""
    if not text or not isinstance(text, str):
        return None
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return None
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else None


def validate_message(msg):
    """Проверяет сообщение на валидность"""
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует поле '{field}'")
        elif msg[field] is None:
            errors.append(f"ValueError: Поле '{field}' имеет значение None")
    
    if errors:
        return (None, errors)
    
    valid_msg = msg.copy()
    
    # Проверка ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину")
    else:
        valid_msg['publisher_inn'] = inn
    
    # Проверка msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # Проверка date_published
    parsed_date = parse_date(msg.get('date_published'))
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # Проверка type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # Проверка case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================

with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)


# ============================================
# 3. ВАЛИДАЦИЯ И ОБОГАЩЕНИЕ СООБЩЕНИЙ
# ============================================

# Создаём словарь организаций
org_by_inn = {org['inn']: org for org in organizations}

# Валидируем и обогащаем сообщения
valid_messages = []
error_records = []

for msg in messages:
    # Сначала валидируем сообщение
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        # Добавляем информацию об организации
        inn = valid_msg.get('publisher_inn')
        org_data = org_by_inn.get(inn)
        
        if org_data:
            valid_msg['org_name'] = org_data.get('name')
            valid_msg['region'] = org_data.get('region')
        else:
            valid_msg['org_name'] = None
            valid_msg['region'] = None
        
        # ДОБАВЛЯЕМ НОВЫЕ ПОЛЯ
        text = valid_msg.get('msg_text', '')
        valid_msg['amounts'] = extract_amounts(text)
        valid_msg['has_court_mention'] = find_court_mentions(text)
        valid_msg['manager_name'] = extract_manager_name(text)
        
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'original_msg': msg,
            'errors': errors
        })


# ============================================
# 4. ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("="*70)
print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ И ОБОГАЩЕНИЯ")
print("="*70)
print(f" Валидных сообщений: {len(valid_messages)}")
print(f" Невалидных сообщений: {len(error_records)}")
print(f" Процент валидных: {len(valid_messages) / len(messages) * 100:.1f}%")


# ============================================
# 5. ПРИМЕРЫ ОБОГАЩЁННЫХ СООБЩЕНИЙ
# ============================================

print("\n" + "="*70)
print("ПРИМЕРЫ ОБОГАЩЁННЫХ СООБЩЕНИЙ (первые 3)")
print("="*70)

for i, msg in enumerate(valid_messages[:3]):
    print(f"\n ОБОГАЩЁННОЕ СООБЩЕНИЕ {i+1}:")
    print(f"   ИНН: {msg.get('publisher_inn')}")
    print(f"   Организация: {msg.get('org_name')}")
    print(f"   Регион: {msg.get('region')}")
    print(f"   Тип: {msg.get('type')}")
    print(f"   Номер дела: {msg.get('case_number')}")
    print(f"   Дата: {msg.get('date_published')}")
    print(f"   Распарсеная дата: {msg.get('parsed_date')}")
    print(f"    Суммы: {msg.get('amounts')}")
    print(f"    Упоминание суда: {msg.get('has_court_mention')}")
    print(f"    Управляющий: {msg.get('manager_name')}")
    print(f"    Текст: {msg.get('msg_text')[:100]}...")


# ============================================
# 6. СТАТИСТИКА ПО НОВЫМ ПОЛЯМ
# ============================================

print("\n" + "="*70)
print("СТАТИСТИКА ПО ОБОГАЩЁННЫМ ПОЛЯМ")
print("="*70)

# Статистика по суммам
messages_with_amounts = sum(1 for msg in valid_messages if msg['amounts'])
total_amounts = sum(len(msg['amounts']) for msg in valid_messages)

print(f"\n СУММЫ:")
print(f"   Сообщений с суммами: {messages_with_amounts}")
print(f"   Всего найдено сумм: {total_amounts}")

# Статистика по судам
messages_with_court = sum(1 for msg in valid_messages if msg['has_court_mention'])
print(f"\n СУДЫ:")
print(f"   Сообщений с упоминанием суда: {messages_with_court}")
print(f"   Процент: {messages_with_court / len(valid_messages) * 100:.1f}%")

# Статистика по управляющим
messages_with_manager = sum(1 for msg in valid_messages if msg['manager_name'])
unique_managers = set(msg['manager_name'] for msg in valid_messages if msg['manager_name'])

print(f"\n УПРАВЛЯЮЩИЕ:")
print(f"   Сообщений с ФИО управляющего: {messages_with_manager}")
print(f"   Уникальных управляющих: {len(unique_managers)}")
print(f"   Процент: {messages_with_manager / len(valid_messages) * 100:.1f}%")


# ============================================
# 7. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================

# Подготавливаем данные для сохранения (преобразуем datetime в строку)
messages_to_save = []
for msg in valid_messages:
    msg_copy = msg.copy()
    if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
        msg_copy['parsed_date'] = msg_copy['parsed_date'].isoformat()
    messages_to_save.append(msg_copy)

with open('valid_messages_enriched.json', 'w', encoding='utf-8') as f:
    json.dump(messages_to_save, f, ensure_ascii=False, indent=2)

print(f"\n Файл сохранён: valid_messages_enriched.json")


# ============================================
# 8. ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ
# ============================================

print("\n" + "="*70)
print("ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ")
print("="*70)

# Топ-3 управляющих по частоте упоминания
from collections import Counter
manager_counts = Counter(msg['manager_name'] for msg in valid_messages if msg['manager_name'])

if manager_counts:
    print("\n Топ-3 арбитражных управляющих:")
    for i, (manager, count) in enumerate(manager_counts.most_common(3)):
        print(f"   {i+1}. {manager} (упоминается {count} раз)")

# Типы сообщений, в которых чаще всего встречаются суммы
print("\n Типы сообщений с суммами:")
type_amounts = Counter()
for msg in valid_messages:
    if msg['amounts']:
        type_amounts[msg['type']] += 1

for msg_type, count in type_amounts.most_common(5):
    print(f"   {msg_type}: {count} сообщений")

РЕЗУЛЬТАТЫ ВАЛИДАЦИИ И ОБОГАЩЕНИЯ
 Валидных сообщений: 48
 Невалидных сообщений: 6
 Процент валидных: 88.9%

ПРИМЕРЫ ОБОГАЩЁННЫХ СООБЩЕНИЙ (первые 3)

 ОБОГАЩЁННОЕ СООБЩЕНИЕ 1:
   ИНН: 7701234567
   Организация: ООО "Альфа-Строй"
   Регион: Москва
   Тип: Введение процедуры
   Номер дела: А60-12345/2025
   Дата: 15.01.2025
   Распарсеная дата: 2025-01-15 00:00:00
    Суммы: ['15 000 000 руб.']
    Упоминание суда: True
    Управляющий: Иванов Иван Иванович.
    Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Ива...

 ОБОГАЩЁННОЕ СООБЩЕНИЕ 2:
   ИНН: 7702345678
   Организация: ЗАО "Бета-Инвест"
   Регион: Москва
   Тип: Собрание кредиторов
   Номер дела: А60-56789/2025
   Дата: 2025-02-10
   Распарсеная дата: 2025-02-10 00:00:00
    Суммы: ['8 500 тыс. руб.']
    Упоминание суда: True
    Управляющий: None
    Текст: Уведомление о проведении собрания кредиторов ЗАО "Бета-Инвест". Сумма требований: 8 500 тыс. руб. АС...

 ОБОГАЩЁННО

### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [43]:
# Ваш код

import json
import re
from datetime import datetime
from collections import Counter

# ============================================
# 1. ВСЕ НЕОБХОДИМЫЕ ФУНКЦИИ
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def extract_amounts(text):
    """Ищет упоминания денежных сумм в тексте"""
    if not text or not isinstance(text, str):
        return []
    
    amounts = []
    
    patterns = [
        (r'(\d{1,3}(?:\s\d{3})*)\s+млн\s+руб\.', 'млн руб.'),
        (r'(\d{1,3}(?:\s\d{3})*)\s+тыс\.\s+руб\.', 'тыс. руб.'),
        (r'(\d{1,3}(?:\s\d{3})*(?:\s\d+)?)\s+руб\.', 'руб.')
    ]
    
    for pattern, suffix in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            amounts.append(f"{match} {suffix}")
    
    return amounts


def find_court_mentions(text):
    """Проверяет упоминание арбитражного суда"""
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


def extract_manager_name(text):
    """Ищет ФИО арбитражного управляющего"""
    if not text or not isinstance(text, str):
        return ""
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return ""
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else ""


def validate_message(msg):
    """Проверяет сообщение на валидность"""
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует поле '{field}'")
    
    if errors:
        return (None, errors)
    
    valid_msg = msg.copy()
    
    # Проверка ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину")
    else:
        valid_msg['publisher_inn'] = inn
    
    # Проверка msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # Проверка date_published
    parsed_date = parse_date(msg.get('date_published'))
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # Проверка type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # Проверка case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================

with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)


# ============================================
# 3. ВАЛИДАЦИЯ И ОБОГАЩЕНИЕ СООБЩЕНИЙ (БЕЗ NONE)
# ============================================

# Создаём словарь организаций
org_by_inn = {org['inn']: org for org in organizations}

# Валидируем и обогащаем сообщения
valid_messages = []
error_records = []

for msg in messages:
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        # Добавляем информацию об организации (заменяем None на "Неизвестно")
        inn = valid_msg.get('publisher_inn')
        org_data = org_by_inn.get(inn)
        
        if org_data:
            valid_msg['org_name'] = org_data.get('name', 'Неизвестно')
            valid_msg['region'] = org_data.get('region', 'Неизвестно')
        else:
            valid_msg['org_name'] = 'Неизвестно'
            valid_msg['region'] = 'Неизвестно'
        
        # Добавляем новые поля (без None)
        text = valid_msg.get('msg_text', '')
        valid_msg['amounts'] = extract_amounts(text)
        valid_msg['has_court_mention'] = find_court_mentions(text)
        valid_msg['manager_name'] = extract_manager_name(text) or 'Не указан'
        
        # Заменяем None в других полях на безопасные значения
        if valid_msg.get('type') is None:
            valid_msg['type'] = 'Не указан'
        if valid_msg.get('case_number') is None:
            valid_msg['case_number'] = 'Не указан'
        if valid_msg.get('publisher_inn') is None:
            valid_msg['publisher_inn'] = 'Не указан'
        
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'original_msg': msg,
            'errors': errors
        })


# ============================================
# 4. ВЫВОД РЕЗУЛЬТАТОВ
# ============================================

print("="*70)
print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ И ОБОГАЩЕНИЯ (БЕЗ NONE)")
print("="*70)
print(f" Валидных сообщений: {len(valid_messages)}")
print(f" Невалидных сообщений: {len(error_records)}")
print(f" Процент валидных: {len(valid_messages) / len(messages) * 100:.1f}%")


# ============================================
# 5. ПОКАЗАТЕЛЬ 1: КОЛИЧЕСТВО СООБЩЕНИЙ ПО ТИПАМ
# ============================================

print("\n" + "="*70)
print(" ПОКАЗАТЕЛЬ 1: КОЛИЧЕСТВО СООБЩЕНИЙ ПО ТИПАМ")
print("="*70)

type_counts = Counter(msg['type'] for msg in valid_messages)

print(f"\nВсего типов: {len(type_counts)}")
print("\n{:<35} {:>10}".format("Тип сообщения", "Количество"))
print("-" * 47)
for msg_type, count in type_counts.most_common():
    print("{:<35} {:>10}".format(msg_type, count))

print("\n Гистограмма:")
for msg_type, count in type_counts.most_common(10):
    bar = "█" * min(count, 20)
    print(f"   {msg_type[:25]:25} | {bar} {count}")


# ============================================
# 6. ПОКАЗАТЕЛЬ 2: КОЛИЧЕСТВО СООБЩЕНИЙ ПО РЕГИОНАМ
# ============================================

print("\n" + "="*70)
print(" ПОКАЗАТЕЛЬ 2: КОЛИЧЕСТВО СООБЩЕНИЙ ПО РЕГИОНАМ")
print("="*70)

# Фильтруем сообщения, где регион не "Неизвестно"
messages_with_region = [msg for msg in valid_messages if msg['region'] != 'Неизвестно']
region_counts = Counter(msg['region'] for msg in messages_with_region)

print(f"\nВсего сообщений с известным регионом: {len(messages_with_region)} из {len(valid_messages)}")
print(f"Пропущено (регион неизвестен): {len(valid_messages) - len(messages_with_region)}")
print(f"\nВсего уникальных регионов: {len(region_counts)}")

print("\n{:<40} {:>10}".format("Регион", "Количество"))
print("-" * 52)
for region, count in region_counts.most_common():
    print("{:<40} {:>10}".format(region, count))

print("\n Гистограмма:")
for region, count in region_counts.most_common(10):
    bar = "█" * min(count, 15)
    print(f"   {region[:35]:35} | {bar} {count}")


# ============================================
# 7. ПОКАЗАТЕЛЬ 3: ТОП-5 ПУБЛИКАТОРОВ
# ============================================

print("\n" + "="*70)
print(" ПОКАЗАТЕЛЬ 3: ТОП-5 ПУБЛИКАТОРОВ ПО КОЛИЧЕСТВУ СООБЩЕНИЙ")
print("="*70)

publisher_counts = Counter(msg['publisher_inn'] for msg in valid_messages)

publisher_stats = []
for inn, count in publisher_counts.items():
    if inn != 'Не указан':
        # Находим организацию по ИНН
        org_name = 'Неизвестно'
        for msg in valid_messages:
            if msg['publisher_inn'] == inn and msg['org_name'] != 'Неизвестно':
                org_name = msg['org_name']
                break
        publisher_stats.append({'inn': inn, 'org_name': org_name, 'count': count})

top_publishers = sorted(publisher_stats, key=lambda x: x['count'], reverse=True)[:5]

print("\n{:<15} {:<35} {:>10}".format("ИНН", "Организация", "Сообщений"))
print("-" * 62)
for publisher in top_publishers:
    print("{:<15} {:<35} {:>10}".format(
        publisher['inn'], 
        publisher['org_name'][:35], 
        publisher['count']
    ))

if top_publishers:
    total_top = sum(p['count'] for p in top_publishers)
    print(f"\n Топ-5 публикаторов отправили {total_top} сообщений")
    print(f"   Это {total_top / len(valid_messages) * 100:.1f}% от всех валидных сообщений")


# ============================================
# 8. ПОКАЗАТЕЛЬ 4: ТАЙМЛАЙН СООБЩЕНИЙ
# ============================================

print("\n" + "="*70)
print(" ПОКАЗАТЕЛЬ 4: ТАЙМЛАЙН СООБЩЕНИЙ")
print("="*70)

# Фильтруем сообщения с корректной датой
messages_with_date = [msg for msg in valid_messages if msg.get('parsed_date') is not None]

# Сортируем по дате
timeline = sorted(messages_with_date, key=lambda x: x['parsed_date'])

print(f"\nВсего сообщений с корректной датой: {len(timeline)}")
if timeline:
    print(f"Самое раннее сообщение: {timeline[0]['parsed_date'].date()}")
    print(f"Самое позднее сообщение: {timeline[-1]['parsed_date'].date()}")

# Группировка по месяцам
print("\n Сообщения по месяцам:")
monthly_counts = Counter()
for msg in timeline:
    month_key = msg['parsed_date'].strftime('%Y-%m')
    monthly_counts[month_key] += 1

print("\n{:<10} {:>10} {:>15}".format("Месяц", "Сообщений", "Накопленный итог"))
print("-" * 40)
cumulative = 0
for month in sorted(monthly_counts.keys()):
    cumulative += monthly_counts[month]
    print("{:<10} {:>10} {:>15}".format(month, monthly_counts[month], cumulative))

# Визуализация
print("\n Таймлайн по месяцам (гистограмма):")
if monthly_counts:
    max_count = max(monthly_counts.values())
    for month in sorted(monthly_counts.keys()):
        count = monthly_counts[month]
        bar_length = int(count / max_count * 20) if max_count > 0 else 0
        bar = "█" * bar_length
        print(f"   {month} | {bar} {count}")

# Первые 10 сообщений
print("\n ПЕРВЫЕ 10 СООБЩЕНИЙ (самые старые):")
print("-" * 80)
for i, msg in enumerate(timeline[:10]):
    date_str = msg['parsed_date'].strftime('%d.%m.%Y')
    org_name = msg.get('org_name', 'Неизвестно')[:35]
    msg_type = msg.get('type', 'Не указан')[:25]
    print(f"{i+1:2}. {date_str} | {org_name:35} | {msg_type}")

# Последние 10 сообщений
print("\n ПОСЛЕДНИЕ 10 СООБЩЕНИЙ (самые новые):")
print("-" * 80)
for i, msg in enumerate(timeline[-10:], 1):
    date_str = msg['parsed_date'].strftime('%d.%m.%Y')
    org_name = msg.get('org_name', 'Неизвестно')[:35]
    msg_type = msg.get('type', 'Не указан')[:25]
    print(f"{i:2}. {date_str} | {org_name:35} | {msg_type}")


# ============================================
# 9. ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ
# ============================================

print("\n" + "="*70)
print(" ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ")
print("="*70)

# Статистика по суммам
messages_with_amounts = sum(1 for msg in valid_messages if msg['amounts'])
total_amounts = sum(len(msg['amounts']) for msg in valid_messages)

print(f"\n СУММЫ:")
print(f"   Сообщений с суммами: {messages_with_amounts}")
print(f"   Всего найдено сумм: {total_amounts}")

# Статистика по судам
messages_with_court = sum(1 for msg in valid_messages if msg['has_court_mention'])
print(f"\n СУДЫ:")
print(f"   Сообщений с упоминанием суда: {messages_with_court}")
print(f"   Процент: {messages_with_court / len(valid_messages) * 100:.1f}%")

# Статистика по управляющим
messages_with_manager = sum(1 for msg in valid_messages if msg['manager_name'] != 'Не указан')
unique_managers = set(msg['manager_name'] for msg in valid_messages if msg['manager_name'] != 'Не указан')

print(f"\n УПРАВЛЯЮЩИЕ:")
print(f"   Сообщений с ФИО управляющего: {messages_with_manager}")
print(f"   Уникальных управляющих: {len(unique_managers)}")
print(f"   Процент: {messages_with_manager / len(valid_messages) * 100:.1f}%")

# Топ-3 управляющих
if unique_managers:
    manager_counts = Counter(msg['manager_name'] for msg in valid_messages if msg['manager_name'] != 'Не указан')
    print(f"\n Топ-3 арбитражных управляющих:")
    for i, (manager, count) in enumerate(manager_counts.most_common(3)):
        print(f"   {i+1}. {manager} (упоминается {count} раз)")


# ============================================
# 10. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================

# Подготавливаем данные для сохранения
messages_to_save = []
for msg in valid_messages:
    msg_copy = msg.copy()
    if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
        msg_copy['parsed_date'] = msg_copy['parsed_date'].isoformat()
    messages_to_save.append(msg_copy)

with open('valid_messages_enriched.json', 'w', encoding='utf-8') as f:
    json.dump(messages_to_save, f, ensure_ascii=False, indent=2)

print(f"\n Файл сохранён: valid_messages_enriched.json")

# Сохраняем результаты анализа
results = {
    "total_valid_messages": len(valid_messages),
    "type_counts": dict(type_counts),
    "region_counts": dict(region_counts),
    "top_publishers": [
        {"inn": p['inn'], "org_name": p['org_name'], "count": p['count']}
        for p in top_publishers
    ],
    "monthly_counts": dict(monthly_counts)
}

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(" Файл сохранён: analysis_results.json")

print("\n" + "="*70)
print(" АНАЛИЗ ЗАВЕРШЕН!")
print("="*70)

РЕЗУЛЬТАТЫ ВАЛИДАЦИИ И ОБОГАЩЕНИЯ (БЕЗ NONE)
 Валидных сообщений: 48
 Невалидных сообщений: 6
 Процент валидных: 88.9%

 ПОКАЗАТЕЛЬ 1: КОЛИЧЕСТВО СООБЩЕНИЙ ПО ТИПАМ

Всего типов: 14

Тип сообщения                       Количество
-----------------------------------------------
Введение процедуры                           8
Завершение процедуры                         6
Продажа имущества                            6
Требование кредитора                         4
Собрание кредиторов                          3
Признание банкротом                          3
Оспаривание сделки                           3
Подача заявления                             3
Мировое соглашение                           3
Передача дела                                3
Субсидиарная ответственность                 2
Назначение управляющего                      2
Отстранение управляющего                     1
Жалоба на управляющего                       1

 Гистограмма:
   Введение процедуры        | ████████ 8
   Заве

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [44]:
# Ваш код

import json
import re
from datetime import datetime
from collections import Counter

# ============================================
# 1. ВСЕ НЕОБХОДИМЫЕ ФУНКЦИИ
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def extract_amounts(text):
    """Ищет упоминания денежных сумм в тексте"""
    if not text or not isinstance(text, str):
        return []
    
    amounts = []
    
    patterns = [
        (r'(\d{1,3}(?:\s\d{3})*)\s+млн\s+руб\.', 'млн руб.'),
        (r'(\d{1,3}(?:\s\d{3})*)\s+тыс\.\s+руб\.', 'тыс. руб.'),
        (r'(\d{1,3}(?:\s\d{3})*(?:\s\d+)?)\s+руб\.', 'руб.')
    ]
    
    for pattern, suffix in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            amounts.append(f"{match} {suffix}")
    
    return amounts


def find_court_mentions(text):
    """Проверяет упоминание арбитражного суда"""
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


def extract_manager_name(text):
    """Ищет ФИО арбитражного управляющего"""
    if not text or not isinstance(text, str):
        return ""
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return ""
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else ""


def validate_message(msg):
    """Проверяет сообщение на валидность"""
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует поле '{field}'")
    
    if errors:
        return (None, errors)
    
    valid_msg = msg.copy()
    
    # Проверка ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину")
    else:
        valid_msg['publisher_inn'] = inn
    
    # Проверка msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # Проверка date_published
    parsed_date = parse_date(msg.get('date_published'))
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # Проверка type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # Проверка case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================

with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)


# ============================================
# 3. ВАЛИДАЦИЯ И ОБОГАЩЕНИЕ СООБЩЕНИЙ
# ============================================

# Создаём словарь организаций
org_by_inn = {org['inn']: org for org in organizations}

# Валидируем и обогащаем сообщения
valid_messages = []
error_records = []

for msg in messages:
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        # Добавляем информацию об организации
        inn = valid_msg.get('publisher_inn')
        org_data = org_by_inn.get(inn)
        
        if org_data:
            valid_msg['org_name'] = org_data.get('name', 'Неизвестно')
            valid_msg['region'] = org_data.get('region', 'Неизвестно')
        else:
            valid_msg['org_name'] = 'Неизвестно'
            valid_msg['region'] = 'Неизвестно'
        
        # Добавляем новые поля
        text = valid_msg.get('msg_text', '')
        valid_msg['amounts'] = extract_amounts(text)
        valid_msg['has_court_mention'] = find_court_mentions(text)
        valid_msg['manager_name'] = extract_manager_name(text) or 'Не указан'
        
        # Заменяем None на безопасные значения
        if valid_msg.get('type') is None:
            valid_msg['type'] = 'Не указан'
        if valid_msg.get('case_number') is None:
            valid_msg['case_number'] = 'Не указан'
        if valid_msg.get('publisher_inn') is None:
            valid_msg['publisher_inn'] = 'Не указан'
        
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'original_msg': msg,
            'errors': errors
        })


# ============================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ JSON (ПРЕОБРАЗОВАНИЕ ДАТ)
# ============================================

def prepare_for_json(messages_list):
    """
    Подготавливает список сообщений для сохранения в JSON.
    Преобразует datetime объекты в строки.
    """
    prepared_messages = []
    
    for msg in messages_list:
        # Создаём копию сообщения
        msg_copy = msg.copy()
        
        # Преобразуем parsed_date из datetime в строку
        if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
            if isinstance(msg_copy['parsed_date'], datetime):
                # Формат: "2025-01-15 14:30:00"
                msg_copy['parsed_date'] = msg_copy['parsed_date'].strftime('%Y-%m-%d %H:%M:%S')
        
        # Также преобразуем другие возможные поля с датами
        if 'date_published' in msg_copy and msg_copy['date_published']:
            # Оставляем как есть (это уже строка)
            pass
        
        prepared_messages.append(msg_copy)
    
    return prepared_messages


# Подготавливаем данные для JSON
messages_for_json = prepare_for_json(valid_messages)

print("="*70)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ JSON")
print("="*70)
print(f" Подготовлено сообщений: {len(messages_for_json)}")
print(f" Тип поля 'parsed_date' после преобразования: {type(messages_for_json[0]['parsed_date']) if messages_for_json else 'Нет данных'}")


# ============================================
# 5. СОХРАНЕНИЕ В JSON
# ============================================

# 5.1. Сохраняем обогащённые сообщения
with open('valid_messages_enriched.json', 'w', encoding='utf-8') as f:
    json.dump(messages_for_json, f, ensure_ascii=False, indent=2)

print(f"\n Файл сохранён: valid_messages_enriched.json")
print(f"   Размер файла: {len(json.dumps(messages_for_json, ensure_ascii=False))} символов")


# ============================================
# 6. ПОДГОТОВКА АНАЛИТИЧЕСКИХ ДАННЫХ ДЛЯ JSON
# ============================================

# Считаем статистику для аналитики
type_counts = Counter(msg['type'] for msg in valid_messages)

# Регионы (исключая 'Неизвестно')
messages_with_known_region = [msg for msg in valid_messages if msg['region'] != 'Неизвестно']
region_counts = Counter(msg['region'] for msg in messages_with_known_region)

# Топ-5 публикаторов
publisher_counts = Counter(msg['publisher_inn'] for msg in valid_messages if msg['publisher_inn'] != 'Не указан')
publisher_stats = []
for inn, count in publisher_counts.most_common(5):
    org_name = 'Неизвестно'
    for msg in valid_messages:
        if msg['publisher_inn'] == inn and msg['org_name'] != 'Неизвестно':
            org_name = msg['org_name']
            break
    publisher_stats.append({'inn': inn, 'org_name': org_name, 'count': count})

# Таймлайн (сообщения с датами)
messages_with_date = [msg for msg in valid_messages if msg.get('parsed_date')]
timeline = sorted(messages_with_date, key=lambda x: x['parsed_date'])

# Подготавливаем таймлайн для JSON
timeline_for_json = []
for msg in timeline:
    timeline_for_json.append({
        'date': msg['parsed_date'].strftime('%Y-%m-%d %H:%M:%S') if isinstance(msg['parsed_date'], datetime) else msg['parsed_date'],
        'type': msg.get('type'),
        'org_name': msg.get('org_name'),
        'publisher_inn': msg.get('publisher_inn'),
        'case_number': msg.get('case_number'),
        'amounts': msg.get('amounts'),
        'has_court_mention': msg.get('has_court_mention'),
        'manager_name': msg.get('manager_name')
    })

# Группировка по месяцам
monthly_counts = Counter()
for msg in messages_with_date:
    if isinstance(msg['parsed_date'], datetime):
        month_key = msg['parsed_date'].strftime('%Y-%m')
    else:
        month_key = 'unknown'
    monthly_counts[month_key] += 1

# Собираем все результаты в один словарь
analysis_results = {
    "metadata": {
        "total_valid_messages": len(valid_messages),
        "total_error_records": len(error_records),
        "analysis_date": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    },
    "type_counts": dict(type_counts),
    "region_counts": dict(region_counts),
    "top_publishers": publisher_stats,
    "monthly_counts": dict(monthly_counts),
    "timeline": timeline_for_json,
    "statistics": {
        "messages_with_amounts": sum(1 for msg in valid_messages if msg['amounts']),
        "total_amounts_found": sum(len(msg['amounts']) for msg in valid_messages),
        "messages_with_court_mention": sum(1 for msg in valid_messages if msg['has_court_mention']),
        "messages_with_manager": sum(1 for msg in valid_messages if msg['manager_name'] != 'Не указан'),
        "unique_managers": len(set(msg['manager_name'] for msg in valid_messages if msg['manager_name'] != 'Не указан'))
    }
}

# Сохраняем аналитические результаты
with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print(f" Файл сохранён: analysis_results.json")


# ============================================
# 7. ПРОВЕРКА ЧИТАЕМОСТИ JSON
# ============================================

print("\n" + "="*70)
print("ПРОВЕРКА ЧИТАЕМОСТИ JSON")
print("="*70)

# Показываем первые 3 сообщения в читаемом виде
print("\n Пример первого сообщения из valid_messages_enriched.json:")
if messages_for_json:
    first_msg = messages_for_json[0]
    for key, value in first_msg.items():
        if key == 'msg_text' and len(str(value)) > 100:
            print(f"   {key}: {str(value)[:100]}...")
        else:
            print(f"   {key}: {value}")

print("\n Пример аналитических данных из analysis_results.json:")
print(f"   Всего валидных сообщений: {analysis_results['metadata']['total_valid_messages']}")
print(f"   Типы сообщений: {len(analysis_results['type_counts'])}")
print(f"   Регионы: {len(analysis_results['region_counts'])}")
print(f"   Месяцев в данных: {len(analysis_results['monthly_counts'])}")


# ============================================
# 8. ВЫВОД СТАТИСТИКИ
# ============================================

print("\n" + "="*70)
print("СТАТИСТИКА ПО ПОДГОТОВЛЕННЫМ ДАННЫМ")
print("="*70)

print(f"\n Валидных сообщений: {len(valid_messages)}")
print(f" Невалидных сообщений: {len(error_records)}")
print(f" Процент валидных: {len(valid_messages) / len(messages) * 100:.1f}%")

print(f"\n Сообщений с суммами: {analysis_results['statistics']['messages_with_amounts']}")
print(f" Всего найдено сумм: {analysis_results['statistics']['total_amounts_found']}")

print(f"\n Сообщений с упоминанием суда: {analysis_results['statistics']['messages_with_court_mention']}")
print(f" Сообщений с ФИО управляющего: {analysis_results['statistics']['messages_with_manager']}")
print(f" Уникальных управляющих: {analysis_results['statistics']['unique_managers']}")

print("\n" + "="*70)
print(" ДАННЫЕ УСПЕШНО ПОДГОТОВЛЕНЫ И СОХРАНЕНЫ В JSON!")
print("="*70)

ПОДГОТОВКА ДАННЫХ ДЛЯ JSON
 Подготовлено сообщений: 48
 Тип поля 'parsed_date' после преобразования: <class 'str'>

 Файл сохранён: valid_messages_enriched.json
   Размер файла: 21542 символов
 Файл сохранён: analysis_results.json

ПРОВЕРКА ЧИТАЕМОСТИ JSON

 Пример первого сообщения из valid_messages_enriched.json:
   publisher_inn: 7701234567
   msg_text: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Ива...
   date_published: 15.01.2025
   type: Введение процедуры
   case_number: А60-12345/2025
   parsed_date: 2025-01-15 00:00:00
   org_name: ООО "Альфа-Строй"
   region: Москва
   amounts: ['15 000 000 руб.']
   has_court_mention: True
   manager_name: Иванов Иван Иванович.

 Пример аналитических данных из analysis_results.json:
   Всего валидных сообщений: 48
   Типы сообщений: 14
   Регионы: 8
   Месяцев в данных: 5

СТАТИСТИКА ПО ПОДГОТОВЛЕННЫМ ДАННЫМ

 Валидных сообщений: 48
 Невалидных сообщений: 6
 Процент валидных: 88.9%

 Сооб

### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [45]:
# Ваш код

import json
import re
from datetime import datetime
from collections import Counter

# ============================================
# 1. ВСЕ НЕОБХОДИМЫЕ ФУНКЦИИ
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def extract_amounts(text):
    """Ищет упоминания денежных сумм в тексте"""
    if not text or not isinstance(text, str):
        return []
    
    amounts = []
    
    patterns = [
        (r'(\d{1,3}(?:\s\d{3})*)\s+млн\s+руб\.', 'млн руб.'),
        (r'(\d{1,3}(?:\s\d{3})*)\s+тыс\.\s+руб\.', 'тыс. руб.'),
        (r'(\d{1,3}(?:\s\d{3})*(?:\s\d+)?)\s+руб\.', 'руб.')
    ]
    
    for pattern, suffix in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            amounts.append(f"{match} {suffix}")
    
    return amounts


def find_court_mentions(text):
    """Проверяет упоминание арбитражного суда"""
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


def extract_manager_name(text):
    """Ищет ФИО арбитражного управляющего"""
    if not text or not isinstance(text, str):
        return ""
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return ""
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else ""


def validate_message(msg):
    """Проверяет сообщение на валидность"""
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует поле '{field}'")
    
    if errors:
        return (None, errors)
    
    valid_msg = msg.copy()
    
    # Проверка ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину")
    else:
        valid_msg['publisher_inn'] = inn
    
    # Проверка msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # Проверка date_published
    parsed_date = parse_date(msg.get('date_published'))
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # Проверка type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # Проверка case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================

with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    priority_cases = [line.strip() for line in f if line.strip()]
priority_set = set(priority_cases)


# ============================================
# 3. ВАЛИДАЦИЯ И ОБОГАЩЕНИЕ СООБЩЕНИЙ
# ============================================

# Создаём словарь организаций
org_by_inn = {org['inn']: org for org in organizations}

# Валидируем и обогащаем сообщения
valid_messages = []
error_records = []

for msg in messages:
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        # Добавляем информацию об организации
        inn = valid_msg.get('publisher_inn')
        org_data = org_by_inn.get(inn)
        
        if org_data:
            valid_msg['org_name'] = org_data.get('name', 'Неизвестно')
            valid_msg['region'] = org_data.get('region', 'Неизвестно')
        else:
            valid_msg['org_name'] = 'Неизвестно'
            valid_msg['region'] = 'Неизвестно'
        
        # Добавляем новые поля
        text = valid_msg.get('msg_text', '')
        valid_msg['amounts'] = extract_amounts(text)
        valid_msg['has_court_mention'] = find_court_mentions(text)
        valid_msg['manager_name'] = extract_manager_name(text) or 'Не указан'
        
        # Заменяем None на безопасные значения
        if valid_msg.get('type') is None:
            valid_msg['type'] = 'Не указан'
        if valid_msg.get('case_number') is None:
            valid_msg['case_number'] = 'Не указан'
        if valid_msg.get('publisher_inn') is None:
            valid_msg['publisher_inn'] = 'Не указан'
        
        # Добавляем флаг приоритетности
        valid_msg['is_priority'] = valid_msg.get('case_number', '') in priority_set
        
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'original_msg': msg,
            'errors': errors
        })


# ============================================
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ JSON
# ============================================

def prepare_for_json(messages_list):
    """Подготавливает список сообщений для сохранения в JSON"""
    prepared_messages = []
    
    for msg in messages_list:
        msg_copy = msg.copy()
        
        # Преобразуем parsed_date из datetime в строку
        if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
            if isinstance(msg_copy['parsed_date'], datetime):
                msg_copy['parsed_date'] = msg_copy['parsed_date'].strftime('%Y-%m-%d %H:%M:%S')
        
        prepared_messages.append(msg_copy)
    
    return prepared_messages


# ============================================
# 5. ФОРМИРОВАНИЕ СТРУКТУРЫ ДЛЯ JSON
# ============================================

# Подготавливаем валидные сообщения для JSON
valid_messages_for_json = prepare_for_json(valid_messages)

# 5.1. Статистика по типам сообщений
type_stats = dict(Counter(msg['type'] for msg in valid_messages))

# 5.2. Статистика по регионам (исключая 'Неизвестно')
region_stats_full = Counter(msg['region'] for msg in valid_messages if msg['region'] != 'Неизвестно')
region_stats = dict(region_stats_full)

# 5.3. Приоритетные сообщения (которые есть в priority_cases.txt)
priority_messages = [msg for msg in valid_messages_for_json if msg.get('is_priority', False)]

# Сортируем приоритетные сообщения по дате (если есть)
priority_messages.sort(key=lambda x: x.get('parsed_date', ''), reverse=True)

# Формируем итоговую структуру
analysis_results = {
    "valid_messages": valid_messages_for_json,
    "type_stats": type_stats,
    "region_stats": region_stats,
    "priority_messages": priority_messages
}


# ============================================
# 6. СОХРАНЕНИЕ ФАЙЛА
# ============================================

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print("="*70)
print("СОХРАНЕНИЕ ФАЙЛА analysis_results.json")
print("="*70)
print(f" Файл успешно сохранён!")


# ============================================
# 7. ВЫВОД СТАТИСТИКИ 
# ============================================

print("\n" + "="*70)
print("СОХРАНЕНИЕ ФАЙЛА analysis_results.json")
print("="*70)
print(f" Файл успешно сохранён!")

print(f"\n СТАТИСТИКА СОХРАНЁННЫХ ДАННЫХ:")
print(f"   - Всего валидных сообщений: {len(analysis_results['valid_messages'])}")
print(f"   - Уникальных типов сообщений: {len(analysis_results['type_stats'])}")
print(f"   - Уникальных регионов: {len(analysis_results['region_stats'])}")
print(f"   - Приоритетных сообщений: {len(analysis_results['priority_messages'])}")

print(f"\n СТРУКТУРА ФАЙЛА:")
print("   {")
print(f"     \"valid_messages\": [{len(analysis_results['valid_messages'])} элементов],")
print(f"     \"type_stats\": {len(analysis_results['type_stats'])} записей,")
print(f"     \"region_stats\": {len(analysis_results['region_stats'])} записей,")
print(f"     \"priority_messages\": [{len(analysis_results['priority_messages'])} элементов]")
print("   }")

# Выводим пример содержимого type_stats
print(f"\n type_stats (первые 5):")
for i, (msg_type, count) in enumerate(list(analysis_results['type_stats'].items())[:5]):
    print(f"   {i+1}. {msg_type}: {count}")

# Выводим пример содержимого region_stats
print(f"\n region_stats (первые 5):")
for i, (region, count) in enumerate(list(analysis_results['region_stats'].items())[:5]):
    print(f"   {i+1}. {region}: {count}")

# Выводим примеры приоритетных сообщений
print(f"\n priority_messages (первые 3):")
for i, msg in enumerate(analysis_results['priority_messages'][:3]):
    print(f"   {i+1}. Дело: {msg.get('case_number', 'Нет номера')}")
    print(f"      Организация: {msg.get('org_name', 'Неизвестно')}")
    print(f"      Тип: {msg.get('type', 'Не указан')}")
    print(f"      Дата: {msg.get('parsed_date', 'Не указана')}")
    print()


# ============================================
# 8. ПРОВЕРКА РАЗМЕРА ФАЙЛА
# ============================================

import os
if os.path.exists('analysis_results.json'):
    file_size = os.path.getsize('analysis_results.json')
    if file_size < 1024:
        size_str = f"{file_size} байт"
    elif file_size < 1024 * 1024:
        size_str = f"{file_size / 1024:.2f} КБ"
    else:
        size_str = f"{file_size / (1024 * 1024):.2f} МБ"
    
    print(f" Размер файла: {size_str}")

print("\n" + "="*70)
print(" ГОТОВО! Файл analysis_results.json создан с указанной структурой")
print("="*70)

СОХРАНЕНИЕ ФАЙЛА analysis_results.json
 Файл успешно сохранён!

СОХРАНЕНИЕ ФАЙЛА analysis_results.json
 Файл успешно сохранён!

 СТАТИСТИКА СОХРАНЁННЫХ ДАННЫХ:
   - Всего валидных сообщений: 48
   - Уникальных типов сообщений: 14
   - Уникальных регионов: 8
   - Приоритетных сообщений: 32

 СТРУКТУРА ФАЙЛА:
   {
     "valid_messages": [48 элементов],
     "type_stats": 14 записей,
     "region_stats": 8 записей,
     "priority_messages": [32 элементов]
   }

 type_stats (первые 5):
   1. Введение процедуры: 8
   2. Собрание кредиторов: 3
   3. Завершение процедуры: 6
   4. Признание банкротом: 3
   5. Продажа имущества: 6

 region_stats (первые 5):
   1. Москва: 25
   2. Свердловская область: 6
   3. Санкт-Петербург: 5
   4. Краснодарский край: 3
   5. Челябинская область: 3

 priority_messages (первые 3):
   1. Дело: А60-12345/2025
      Организация: ЗАО "Бета-Инвест"
      Тип: Субсидиарная ответственность
      Дата: 2025-05-25 00:00:00

   2. Дело: А60-77777/2025
      Организация:

### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [46]:
# Ваш код

import json
import re
from datetime import datetime
from collections import Counter

# ============================================
# 1. ВСЕ НЕОБХОДИМЫЕ ФУНКЦИИ
# ============================================

def parse_date(date_str):
    """Парсит строку с датой в разных форматах"""
    if not date_str or not isinstance(date_str, str):
        return None
    
    date_str = date_str.strip()
    if not date_str:
        return None
    
    months_ru = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    
    try:
        if re.match(r'^\d{2}\.\d{2}\.\d{4}$', date_str):
            return datetime.strptime(date_str, '%d.%m.%Y')
        
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return datetime.fromisoformat(date_str)
        
        match = re.match(r'^(\d{1,2})\s+([а-я]+)\s+(\d{4})\s+г\.?$', date_str, re.IGNORECASE)
        if match:
            day = int(match.group(1))
            month_name = match.group(2).lower()
            year = int(match.group(3))
            if month_name in months_ru:
                month = months_ru[month_name]
                return datetime(year, month, day)
        
        if re.match(r'^\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}$', date_str):
            return datetime.strptime(date_str, '%d/%m/%Y %H:%M')
        
        return None
        
    except (ValueError, TypeError):
        return None


def extract_amounts(text):
    """Ищет упоминания денежных сумм в тексте"""
    if not text or not isinstance(text, str):
        return []
    
    amounts = []
    
    patterns = [
        (r'(\d{1,3}(?:\s\d{3})*)\s+млн\s+руб\.', 'млн руб.'),
        (r'(\d{1,3}(?:\s\d{3})*)\s+тыс\.\s+руб\.', 'тыс. руб.'),
        (r'(\d{1,3}(?:\s\d{3})*(?:\s\d+)?)\s+руб\.', 'руб.')
    ]
    
    for pattern, suffix in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            amounts.append(f"{match} {suffix}")
    
    return amounts


def find_court_mentions(text):
    """Проверяет упоминание арбитражного суда"""
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


def extract_manager_name(text):
    """Ищет ФИО арбитражного управляющего"""
    if not text or not isinstance(text, str):
        return ""
    
    keyword = "арбитражный управляющий"
    if keyword.lower() not in text.lower():
        return ""
    
    pos = text.lower().find(keyword.lower())
    after_keyword = text[pos + len(keyword):]
    words = after_keyword.split()
    
    name_parts = words[:3]
    return " ".join(name_parts) if name_parts else ""


def validate_message(msg):
    """Проверяет сообщение на валидность"""
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']
    
    for field in required_fields:
        if field not in msg:
            errors.append(f"KeyError: Отсутствует обязательное поле '{field}'")
    
    if errors:
        return (None, errors)
    
    valid_msg = msg.copy()
    
    # Проверка ИНН
    inn = str(msg['publisher_inn']) if msg['publisher_inn'] is not None else ''
    if not inn:
        errors.append("ValueError: Поле 'publisher_inn' пустое")
    elif not inn.isdigit():
        errors.append(f"ValueError: ИНН '{inn}' должен состоять только из цифр")
    elif len(inn) not in [10, 12]:
        errors.append(f"ValueError: ИНН '{inn}' имеет неверную длину (должен быть 10 или 12 цифр)")
    else:
        valid_msg['publisher_inn'] = inn
    
    # Проверка msg_text
    msg_text = msg.get('msg_text')
    if not msg_text or not isinstance(msg_text, str) or not msg_text.strip():
        errors.append("ValueError: Поле 'msg_text' пустое или не является строкой")
    else:
        valid_msg['msg_text'] = msg_text.strip()
    
    # Проверка date_published
    parsed_date = parse_date(msg.get('date_published'))
    if parsed_date is None:
        errors.append(f"ValueError: Не удалось распарсить дату '{msg.get('date_published')}'")
    else:
        valid_msg['parsed_date'] = parsed_date
    
    # Проверка type
    msg_type = msg.get('type')
    if not msg_type or not isinstance(msg_type, str) or not msg_type.strip():
        errors.append("ValueError: Поле 'type' пустое или не является строкой")
    else:
        valid_msg['type'] = msg_type.strip()
    
    # Проверка case_number
    case_number = msg.get('case_number')
    if not case_number or not isinstance(case_number, str) or not case_number.strip():
        errors.append("ValueError: Поле 'case_number' пустое или не является строкой")
    else:
        valid_msg['case_number'] = case_number.strip()
    
    if errors:
        return (None, errors)
    else:
        return (valid_msg, [])


# ============================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================

with open('organizations.json', 'r', encoding='utf-8') as f:
    organizations = json.load(f)

with open('bankruptcy_messages.json', 'r', encoding='utf-8') as f:
    messages = json.load(f)

with open('priority_cases.txt', 'r', encoding='utf-8') as f:
    priority_cases = [line.strip() for line in f if line.strip()]
priority_set = set(priority_cases)


# ============================================
# 3. ВАЛИДАЦИЯ И ОБОГАЩЕНИЕ СООБЩЕНИЙ
# ============================================

# Создаём словарь организаций
org_by_inn = {org['inn']: org for org in organizations}

# Валидируем и обогащаем сообщения
valid_messages = []
error_records = []

for idx, msg in enumerate(messages):
    valid_msg, errors = validate_message(msg)
    
    if valid_msg:
        # Добавляем информацию об организации
        inn = valid_msg.get('publisher_inn')
        org_data = org_by_inn.get(inn)
        
        if org_data:
            valid_msg['org_name'] = org_data.get('name', 'Неизвестно')
            valid_msg['region'] = org_data.get('region', 'Неизвестно')
        else:
            valid_msg['org_name'] = 'Неизвестно'
            valid_msg['region'] = 'Неизвестно'
        
        # Добавляем новые поля
        text = valid_msg.get('msg_text', '')
        valid_msg['amounts'] = extract_amounts(text)
        valid_msg['has_court_mention'] = find_court_mentions(text)
        valid_msg['manager_name'] = extract_manager_name(text) or 'Не указан'
        
        # Заменяем None на безопасные значения
        if valid_msg.get('type') is None:
            valid_msg['type'] = 'Не указан'
        if valid_msg.get('case_number') is None:
            valid_msg['case_number'] = 'Не указан'
        if valid_msg.get('publisher_inn') is None:
            valid_msg['publisher_inn'] = 'Не указан'
        
        # Добавляем флаг приоритетности
        valid_msg['is_priority'] = valid_msg.get('case_number', '') in priority_set
        
        valid_messages.append(valid_msg)
    else:
        # Сохраняем проблемные записи с индексом и ошибками
        error_records.append({
            'index': idx,
            'original_message': msg,
            'errors': errors,
            'validation_summary': '; '.join(errors)
        })


# ============================================
# 4. СОХРАНЕНИЕ ПРОБЛЕМНЫХ ЗАПИСЕЙ
# ============================================

# Подготавливаем проблемные записи для JSON (преобразуем datetime)
error_records_for_json = []
for record in error_records:
    record_copy = record.copy()
    
    # Если в оригинальном сообщении есть даты, преобразуем их
    if 'date_published' in record_copy['original_message']:
        # Оставляем как есть (это строка)
        pass
    
    error_records_for_json.append(record_copy)

# Сохраняем проблемные записи в отдельный файл
with open('error_records.json', 'w', encoding='utf-8') as f:
    json.dump(error_records_for_json, f, ensure_ascii=False, indent=2)

# ============================================
# 5. ПОДГОТОВКА ДАННЫХ ДЛЯ JSON
# ============================================

def prepare_for_json(messages_list):
    """Подготавливает список сообщений для сохранения в JSON"""
    prepared_messages = []
    
    for msg in messages_list:
        msg_copy = msg.copy()
        
        # Преобразуем parsed_date из datetime в строку
        if 'parsed_date' in msg_copy and msg_copy['parsed_date']:
            if isinstance(msg_copy['parsed_date'], datetime):
                msg_copy['parsed_date'] = msg_copy['parsed_date'].strftime('%Y-%m-%d %H:%M:%S')
        
        prepared_messages.append(msg_copy)
    
    return prepared_messages


# ============================================
# 6. ФОРМИРОВАНИЕ СТРУКТУРЫ ДЛЯ JSON
# ============================================

# Подготавливаем валидные сообщения для JSON
valid_messages_for_json = prepare_for_json(valid_messages)

# Статистика по типам сообщений
type_stats = dict(Counter(msg['type'] for msg in valid_messages))

# Статистика по регионам (исключая 'Неизвестно')
region_stats_full = Counter(msg['region'] for msg in valid_messages if msg['region'] != 'Неизвестно')
region_stats = dict(region_stats_full)

# Приоритетные сообщения
priority_messages = [msg for msg in valid_messages_for_json if msg.get('is_priority', False)]
priority_messages.sort(key=lambda x: x.get('parsed_date', ''), reverse=True)

# Формируем итоговую структуру
analysis_results = {
    "valid_messages": valid_messages_for_json,
    "type_stats": type_stats,
    "region_stats": region_stats,
    "priority_messages": priority_messages
}


# ============================================
# 7. СОХРАНЕНИЕ ФАЙЛА analysis_results.json
# ============================================

with open('analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)


# ============================================
# 8. ВЫВОД СТАТИСТИКИ
# ============================================

print("="*70)
print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*70)

print(f"\n Валидных сообщений: {len(valid_messages)}")
print(f" Проблемных записей: {len(error_records)}")
print(f" Процент валидных: {len(valid_messages) / len(messages) * 100:.1f}%")

print(f"\n Сохранённые файлы:")
print(f"   1. analysis_results.json - валидные сообщения и статистика")
print(f"   2. error_records.json - проблемные записи с описанием ошибок")

# ============================================
# 9. ВЫВОД ПРИМЕРОВ ПРОБЛЕМНЫХ ЗАПИСЕЙ
# ============================================

print("\n" + "="*70)
print("ПРИМЕРЫ ПРОБЛЕМНЫХ ЗАПИСЕЙ (первые 5)")
print("="*70)

for i, record in enumerate(error_records[:5]):
    print(f"\n ПРОБЛЕМНАЯ ЗАПИСЬ {i+1}:")
    print(f"   Индекс в исходном массиве: {record['index']}")
    print(f"   Краткое описание: {record['validation_summary'][:100]}...")
    print(f"   Детальные ошибки:")
    for error in record['errors']:
        print(f"      - {error}")
    print(f"   Содержимое сообщения:")
    original = record['original_message']
    if original.get('publisher_inn'):
        print(f"      ИНН: {original.get('publisher_inn')}")
    if original.get('type'):
        print(f"      Тип: {original.get('type')}")
    if original.get('case_number'):
        print(f"      Номер дела: {original.get('case_number')}")
    if original.get('date_published'):
        print(f"      Дата: {original.get('date_published')}")

# ============================================
# 10. СТАТИСТИКА ПО ТИПАМ ОШИБОК
# ============================================

print("\n" + "="*70)
print("СТАТИСТИКА ПО ТИПАМ ОШИБОК")
print("="*70)

error_type_counter = Counter()
error_detail_counter = Counter()

for record in error_records:
    for error in record['errors']:
        if error.startswith('KeyError'):
            error_type_counter['KeyError'] += 1
        elif error.startswith('ValueError'):
            error_type_counter['ValueError'] += 1
        else:
            error_type_counter['Other'] += 1
        
        # Детальная статистика
        error_detail_counter[error] += 1

print(f"\n Ошибки по типам:")
for error_type, count in error_type_counter.most_common():
    print(f"   {error_type}: {count}")

print(f"\n Наиболее частые ошибки:")
for error, count in error_detail_counter.most_common(5):
    print(f"   - {error[:80]}... (встретилось {count} раз)")

# ============================================
# 11. ПРОВЕРКА РАЗМЕРОВ ФАЙЛОВ
# ============================================

import os

print("\n" + "="*70)
print("РАЗМЕРЫ СОХРАНЁННЫХ ФАЙЛОВ")
print("="*70)

files = ['analysis_results.json', 'error_records.json']
for file_name in files:
    if os.path.exists(file_name):
        file_size = os.path.getsize(file_name)
        if file_size < 1024:
            size_str = f"{file_size} байт"
        elif file_size < 1024 * 1024:
            size_str = f"{file_size / 1024:.2f} КБ"
        else:
            size_str = f"{file_size / (1024 * 1024):.2f} МБ"
        print(f"   📄 {file_name}: {size_str}")

print("\n" + "="*70)
print(" ГОТОВО! Проблемные записи сохранены в error_records.json")
print("="*70)

СОХРАНЕНИЕ РЕЗУЛЬТАТОВ

 Валидных сообщений: 48
 Проблемных записей: 6
 Процент валидных: 88.9%

 Сохранённые файлы:
   1. analysis_results.json - валидные сообщения и статистика
   2. error_records.json - проблемные записи с описанием ошибок

ПРИМЕРЫ ПРОБЛЕМНЫХ ЗАПИСЕЙ (первые 5)

 ПРОБЛЕМНАЯ ЗАПИСЬ 1:
   Индекс в исходном массиве: 47
   Краткое описание: ValueError: Поле 'publisher_inn' пустое...
   Детальные ошибки:
      - ValueError: Поле 'publisher_inn' пустое
   Содержимое сообщения:
      Тип: Подача заявления
      Номер дела: А60-24680/2025
      Дата: 2025-04-25

 ПРОБЛЕМНАЯ ЗАПИСЬ 2:
   Индекс в исходном массиве: 49
   Краткое описание: ValueError: Не удалось распарсить дату 'None'...
   Детальные ошибки:
      - ValueError: Не удалось распарсить дату 'None'
   Содержимое сообщения:
      ИНН: 7701234567
      Тип: Введение процедуры
      Номер дела: А60-36925/2025

 ПРОБЛЕМНАЯ ЗАПИСЬ 3:
   Индекс в исходном массиве: 50
   Краткое описание: ValueError: Поле 'type' пустое и

### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [50]:
# Ваш код

import json
from datetime import datetime
from collections import Counter

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================

# Загружаем результаты анализа
with open('analysis_results.json', 'r', encoding='utf-8') as f:
    analysis_results = json.load(f)

with open('error_records.json', 'r', encoding='utf-8') as f:
    error_records = json.load(f)

# Базовые переменные
valid_messages = analysis_results['valid_messages']
type_stats = analysis_results['type_stats']
region_stats = analysis_results['region_stats']
priority_messages = analysis_results['priority_messages']

total_messages = len(valid_messages) + len(error_records)
valid_count = len(valid_messages)
error_count = len(error_records)

# Собираем уникальных управляющих из валидных сообщений
managers = set()
for msg in valid_messages:
    manager = msg.get('manager_name', '')
    if manager and manager != 'Не указан':
        managers.add(manager)

# Топ-5 публикаторов (собираем из valid_messages)
publisher_counts = Counter()
for msg in valid_messages:
    inn = msg.get('publisher_inn', 'Не указан')
    if inn != 'Не указан':
        publisher_counts[inn] += 1

top_publishers = []
for inn, count in publisher_counts.most_common(5):
    # Находим название организации
    org_name = 'Неизвестно'
    for msg in valid_messages:
        if msg.get('publisher_inn') == inn and msg.get('org_name') != 'Неизвестно':
            org_name = msg.get('org_name')
            break
    top_publishers.append({'inn': inn, 'org_name': org_name, 'count': count})

# ============================================
# 2. ФОРМИРОВАНИЕ ТЕКСТОВОГО ОТЧЁТА
# ============================================

# Текущая дата для отчёта
report_date = datetime.now().strftime('%d.%m.%Y %H:%M:%S')

# Создаём отчёт
report = f"""
{'='*80}
                    АНАЛИТИЧЕСКИЙ ОТЧЁТ
                    ПО БАНКРОТНЫМ СООБЩЕНИЯМ
{'='*80}

Дата формирования отчёта: {report_date}
Период анализа: {'01.01.2025 - 31.05.2025'}

{'='*80}
1. ОБЩАЯ СТАТИСТИКА
{'='*80}

   Всего получено сообщений:          {total_messages}
    Валидных сообщений:              {valid_count}
    Ошибочных сообщений:             {error_count}
    Процент валидных сообщений:      {valid_count / total_messages * 100:.1f}%

   Из них:
   - Сообщений с денежными суммами:   {sum(1 for msg in valid_messages if msg.get('amounts'))}
   - Сообщений с упоминанием суда:    {sum(1 for msg in valid_messages if msg.get('has_court_mention'))}
   - Сообщений с ФИО управляющего:    {sum(1 for msg in valid_messages if msg.get('manager_name') and msg['manager_name'] != 'Не указан')}

{'='*80}
2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
{'='*80}

   Всего уникальных типов: {len(type_stats)}

   { 'Тип сообщения':<35} {'Количество':>10} {'Доля':>10}
   {'-'*57}
"""

# Добавляем типы сообщений
for msg_type, count in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
    percentage = count / valid_count * 100
    report += f"   {msg_type:<35} {count:>10} {percentage:>9.1f}%\n"

report += f"""
{'='*80}
3. СТАТИСТИКА ПО РЕГИОНАМ
{'='*80}

   Всего уникальных регионов: {len(region_stats)}
   Всего сообщений с известным регионом: {sum(region_stats.values())}

   { 'Регион':<40} {'Количество':>10} {'Доля':>10}
   {'-'*62}
"""

# Добавляем регионы
for region, count in sorted(region_stats.items(), key=lambda x: x[1], reverse=True):
    percentage = count / valid_count * 100
    report += f"   {region:<40} {count:>10} {percentage:>9.1f}%\n"

report += f"""
{'='*80}
4. ТОП-5 ПУБЛИКАТОРОВ
{'='*80}

   { 'ИНН':<15} {'Организация':<40} {'Сообщений':>10}
   {'-'*67}
"""

# Добавляем топ-5 публикаторов
for publisher in top_publishers:
    report += f"   {publisher['inn']:<15} {publisher['org_name']:<40} {publisher['count']:>10}\n"

if top_publishers:
    total_by_top = sum(p['count'] for p in top_publishers)
    report += f"\n    Топ-5 публикаторов отправили {total_by_top} сообщений"
    report += f" ({total_by_top / valid_count * 100:.1f}% от всех валидных сообщений)\n"

report += f"""
{'='*80}
5. ПРИОРИТЕТНЫЕ ДЕЛА
{'='*80}

   Всего приоритетных дел: {len(priority_messages)}

   {'№':<4} {'Номер дела':<20} {'Организация':<30} {'Тип сообщения':<25}
   {'-'*85}
"""

# Добавляем приоритетные дела
for i, msg in enumerate(priority_messages[:20], 1):
    case_number = msg.get('case_number', 'Не указан')
    org_name = msg.get('org_name', 'Неизвестно')[:28]
    msg_type = msg.get('type', 'Не указан')[:23]
    report += f"   {i:<4} {case_number:<20} {org_name:<30} {msg_type:<25}\n"

if len(priority_messages) > 20:
    report += f"\n   ... и ещё {len(priority_messages) - 20} приоритетных дел\n"

report += f"""
{'='*80}
6. АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ
{'='*80}

   Всего уникальных управляющих: {len(managers)}

"""

if managers:
    report += "   Список найденных арбитражных управляющих:\n\n"
    for i, manager in enumerate(sorted(managers), 1):
        report += f"   {i:2}. {manager}\n"
else:
    report += "    Арбитражные управляющие не найдены\n"

report += f"""
{'='*80}
7. ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ
{'='*80}

    Временной диапазон:
   - Самое раннее сообщение: {min((msg.get('parsed_date', '') for msg in valid_messages if msg.get('parsed_date')), default='Нет данных')}
   - Самое позднее сообщение: {max((msg.get('parsed_date', '') for msg in valid_messages if msg.get('parsed_date')), default='Нет данных')}

    Финансовая информация:
   - Сообщений с указанием сумм: {sum(1 for msg in valid_messages if msg.get('amounts'))}
   - Всего найдено упоминаний сумм: {sum(len(msg.get('amounts', [])) for msg in valid_messages)}

    Судебная информация:
   - Сообщений с упоминанием суда: {sum(1 for msg in valid_messages if msg.get('has_court_mention'))}
   - Доля сообщений с судом: {sum(1 for msg in valid_messages if msg.get('has_court_mention')) / valid_count * 100:.1f}%

    Проблемные записи:
   - Количество ошибочных сообщений: {error_count}
   - Процент брака: {error_count / total_messages * 100:.1f}%

{'='*80}
                            КОНЕЦ ОТЧЁТА
{'='*80}
"""

# ============================================
# 3. ВЫВОД ОТЧЁТА НА ЭКРАН
# ============================================

print(report)

# ============================================
# 4. СОХРАНЕНИЕ ОТЧЁТА В ТЕКСТОВЫЙ ФАЙЛ
# ============================================

# Сохраняем отчёт в файл
report_filename = f"analytical_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(report_filename, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n Отчёт сохранён в файл: {report_filename}")

# ============================================
# 5. СОХРАНЕНИЕ КРАТКОЙ ВЕРСИИ ОТЧЁТА
# ============================================

# Краткая версия для быстрого ознакомления
short_report = f"""
{'='*60}
КРАТКИЙ АНАЛИТИЧЕСКИЙ ОТЧЁТ
{'='*60}

 КЛЮЧЕВЫЕ ПОКАЗАТЕЛИ:
   • Всего сообщений: {total_messages}
   • Валидных: {valid_count} ({valid_count / total_messages * 100:.1f}%)
   • Ошибочных: {error_count} ({error_count / total_messages * 100:.1f}%)

 ТОП-3 ТИПА СООБЩЕНИЙ:
"""
for i, (msg_type, count) in enumerate(sorted(type_stats.items(), key=lambda x: x[1], reverse=True)[:3], 1):
    short_report += f"   {i}. {msg_type}: {count}\n"

short_report += f"""
 ТОП-3 РЕГИОНА:
"""
for i, (region, count) in enumerate(sorted(region_stats.items(), key=lambda x: x[1], reverse=True)[:3], 1):
    short_report += f"   {i}. {region}: {count}\n"

short_report += f"""
 ТОП-3 ПУБЛИКАТОРА:
"""
for i, publisher in enumerate(top_publishers[:3], 1):
    short_report += f"   {i}. {publisher['org_name']}: {publisher['count']} сообщ.\n"

short_report += f"""
 ПРИОРИТЕТНЫЕ ДЕЛА: {len(priority_messages)}
 АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ: {len(managers)}

{'='*60}
"""

# Сохраняем краткую версию
short_report_filename = f"short_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(short_report_filename, 'w', encoding='utf-8') as f:
    f.write(short_report)

print(f" Краткий отчёт сохранён в файл: {short_report_filename}")

print("\n" + "="*60)
print(" ОТЧЁТ УСПЕШНО СФОРМИРОВАН!")
print("="*60)


                    АНАЛИТИЧЕСКИЙ ОТЧЁТ
                    ПО БАНКРОТНЫМ СООБЩЕНИЯМ

Дата формирования отчёта: 24.04.2026 11:11:33
Период анализа: 01.01.2025 - 31.05.2025

1. ОБЩАЯ СТАТИСТИКА

   Всего получено сообщений:          54
    Валидных сообщений:              48
    Ошибочных сообщений:             6
    Процент валидных сообщений:      88.9%

   Из них:
   - Сообщений с денежными суммами:   39
   - Сообщений с упоминанием суда:    45
   - Сообщений с ФИО управляющего:    10

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ

   Всего уникальных типов: 14

   Тип сообщения                       Количество       Доля
   ---------------------------------------------------------
   Введение процедуры                           8      16.7%
   Завершение процедуры                         6      12.5%
   Продажа имущества                            6      12.5%
   Требование кредитора                         4       8.3%
   Собрание кредиторов                          3       6.2%
   Признание б

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |



import os
print("Текущая папка:", os.getcwd())
print("Содержимое:", os.listdir())

In [49]:
import os 
print("Текущая папка:", os.getcwd()) 
print("Содержимое:", os.listdir())

Текущая папка: C:\jupyter_work\hw5
Содержимое: ['.ipynb_checkpoints', 'analysis_results.json', 'analytical_report_20260422_104513.txt', 'analytical_report_20260424_110631.txt', 'analytical_report_20260424_110711.txt', 'bankruptcy_messages.json', 'error_records.json', 'final_hw_data', 'final_hw_efrsb.ipynb', 'organizations.json', 'priority_cases.txt', 'short_report_20260422_104513.txt', 'short_report_20260424_110631.txt', 'short_report_20260424_110711.txt', 'valid_messages.json', 'valid_messages_enriched.json']
